# Topic Segmentation Pipeline

Splits a transcript into topics using sentence embeddings and cosine similarity.

**Steps:** Split into sentences → Embed → Compare consecutive pairs → Find boundaries → Group into topics

## Setup

In [4]:
import numpy as np
import nltk
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt_tab', quiet=True)
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Step 1: Split VibeVoice segments into sentences

VibeVoice returns segments by speaker turn, not by sentence.
We use NLTK to split each segment into individual sentences,
estimating timestamps based on position within the segment.

In [5]:
# Simulated VibeVoice output
segments = [
    {"start": 0.0, "end": 14.0, "speaker": 0,
     "content": "All right, so here we are, in front of the elephants. Cool thing about these guys is that they have really long trunks. And that's cool."},
    {"start": 14.0, "end": 16.2, "speaker": None, "content": "[Unintelligible Speech]"},
    {"start": 16.2, "end": 19.01, "speaker": 0,
     "content": "And that's pretty much all there is to say."}
]

all_sentences = []
for seg in segments:
    if seg["content"].startswith("["):  # skip [Music], [Unintelligible Speech], etc.
        continue
    sentences = sent_tokenize(seg["content"])
    seg_duration = seg["end"] - seg["start"]
    time_per_sentence = seg_duration / len(sentences) if sentences else 0
    for i, sentence in enumerate(sentences):
        all_sentences.append({
            "text": sentence,
            "start": round(seg["start"] + (i * time_per_sentence), 2),
            "end": round(seg["start"] + ((i + 1) * time_per_sentence), 2),
            "speaker": seg["speaker"],
        })

for s in all_sentences:
    print(f"[{s['start']:.1f}s - {s['end']:.1f}s] Speaker {s['speaker']}: {s['text']}")

[0.0s - 4.7s] Speaker 0: All right, so here we are, in front of the elephants.
[4.7s - 9.3s] Speaker 0: Cool thing about these guys is that they have really long trunks.
[9.3s - 14.0s] Speaker 0: And that's cool.
[16.2s - 19.0s] Speaker 0: And that's pretty much all there is to say.


## Steps 2-5: Full segmentation pipeline

Using a longer lecture transcript to demonstrate topic detection.
The algorithm: embed sentences → cosine similarity between consecutive pairs → percentile cutoff → group.

In [6]:
# Longer transcript with 3 distinct topics
lecture_sentences = [
    # Topic: What are neural networks
    "Today we're going to talk about neural networks.",
    "A neural network is a computing system inspired by the brain.",
    "It consists of layers of interconnected nodes called neurons.",
    "Each connection between neurons has a weight that gets adjusted during training.",
    # Topic: Training process
    "Now let's discuss how we actually train these networks.",
    "Training involves feeding data through the network and comparing the output to the expected result.",
    "The difference between the output and the expected result is called the loss.",
    "We use an algorithm called backpropagation to minimize this loss.",
    "Backpropagation computes gradients that tell us how to adjust each weight.",
    # Topic: Practical tips
    "Let me share some practical tips for training your models.",
    "Always start with a small learning rate and increase it gradually.",
    "Make sure to split your data into training and validation sets.",
    "Overfitting is a common problem where the model memorizes the training data.",
    "You can use techniques like dropout and data augmentation to prevent overfitting.",
]

# Step 2: Embed each sentence into a 384-dim vector
embeddings = model.encode(lecture_sentences)

# Step 3: Cosine similarity between consecutive sentences
similarities = []
for i in range(len(embeddings) - 1):
    # Get vectors for two consecutive sentences
    current = embeddings[i].reshape(1, -1)      # reshape (384,) to (1, 384)
    next_one = embeddings[i + 1].reshape(1, -1)

    # Compare them — returns [[score]]
    similarity_matrix = cosine_similarity(current, next_one)

    # Extract the single number
    score = similarity_matrix[0][0]
    similarities.append(score)

# Step 4: Percentile-based boundary detection
PERCENTILE = 15
cutoff = np.percentile(similarities, PERCENTILE)
boundaries = [i + 1 for i, s in enumerate(similarities) if s < cutoff]

# Step 5: Group sentences into topics
topics = []
start_idx = 0
for boundary in boundaries:
    topics.append(lecture_sentences[start_idx:boundary])
    start_idx = boundary
topics.append(lecture_sentences[start_idx:])

# Display results
print(f"Cutoff ({PERCENTILE}th percentile): {cutoff:.3f}")
print(f"Boundaries at sentences: {boundaries}")
print(f"Number of topics: {len(topics)}\n")

for i, topic in enumerate(topics):
    print(f"{'='*60}")
    print(f"TOPIC {i + 1} ({len(topic)} sentences)")
    print(f"{'='*60}")
    for sentence in topic:
        print(f"  {sentence}")
    print()

Cutoff (15th percentile): 0.383
Boundaries at sentences: [9, 12]
Number of topics: 3

TOPIC 1 (9 sentences)
  Today we're going to talk about neural networks.
  A neural network is a computing system inspired by the brain.
  It consists of layers of interconnected nodes called neurons.
  Each connection between neurons has a weight that gets adjusted during training.
  Now let's discuss how we actually train these networks.
  Training involves feeding data through the network and comparing the output to the expected result.
  The difference between the output and the expected result is called the loss.
  We use an algorithm called backpropagation to minimize this loss.
  Backpropagation computes gradients that tell us how to adjust each weight.

TOPIC 2 (3 sentences)
  Let me share some practical tips for training your models.
  Always start with a small learning rate and increase it gradually.
  Make sure to split your data into training and validation sets.

TOPIC 3 (2 sentences)
  O

## Experiment: Percentile tuning

Lower percentile = fewer topics (only the most obvious breaks).
Higher percentile = more topics (splits on smaller changes).

In [7]:
for pct in [10, 15, 20, 25, 30]:
    cutoff = np.percentile(similarities, pct)
    num_boundaries = sum(1 for s in similarities if s < cutoff)
    print(f"  Percentile {pct}% → cutoff {cutoff:.3f} → {num_boundaries + 1} topics")

  Percentile 10% → cutoff 0.321 → 3 topics
  Percentile 15% → cutoff 0.383 → 3 topics
  Percentile 20% → cutoff 0.405 → 4 topics
  Percentile 25% → cutoff 0.407 → 4 topics
  Percentile 30% → cutoff 0.412 → 5 topics


In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')

# A topic with 6 sentences — some are important, some are filler
topic_sentences = [
    "Neural networks are computing systems inspired by the biological brain.",
    "They consist of layers of interconnected nodes called neurons.",
    "Each connection has a weight that gets adjusted during training.",
    "Um, so yeah, that's basically how it works.",
    "The key insight is that these networks can learn complex patterns from data.",
    "Anyway, let's move on to the next topic.",
]

# Step 1: Embed all sentences
embeddings = model.encode(topic_sentences)

# Step 2: Compute the centroid (average vector for the whole topic)
centroid = np.mean(embeddings, axis=0).reshape(1, -1)

# Step 3: Score each sentence by similarity to the centroid
scores = []
for i, emb in enumerate(embeddings):
    score = cosine_similarity(emb.reshape(1, -1), centroid)[0][0]
    scores.append(score)
    print(f"  Score {score:.3f}: {topic_sentences[i][:70]}")

# Step 4: Pick the top 2 sentences as the summary
num_summary_sentences = 2
top_indices = np.argsort(scores)[-num_summary_sentences:]
top_indices = sorted(top_indices)  # keep original order

print(f"\nSummary:")
for idx in top_indices:
    print(f"  {topic_sentences[idx]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Score 0.721: Neural networks are computing systems inspired by the biological brain
  Score 0.736: They consist of layers of interconnected nodes called neurons.
  Score 0.540: Each connection has a weight that gets adjusted during training.
  Score 0.560: Um, so yeah, that's basically how it works.
  Score 0.738: The key insight is that these networks can learn complex patterns from
  Score 0.342: Anyway, let's move on to the next topic.

Summary:
  They consist of layers of interconnected nodes called neurons.
  The key insight is that these networks can learn complex patterns from data.


In [2]:
def extractive_summary(sentences: list[str], num_sentences: int = 2, redundancy_threshold: float = 0.8) -> list[str]:
    """
    Picks the most representative sentences from a topic.
    Skips sentences that are too similar to already-selected ones.
    """
    if len(sentences) <= num_sentences:
        return sentences

    # Embed all sentences
    embeddings = model.encode(sentences)

    # Compute centroid (average meaning of the topic)
    centroid = np.mean(embeddings, axis=0).reshape(1, -1)

    # Score each sentence by similarity to centroid
    scores = []
    for emb in embeddings:
        score = cosine_similarity(emb.reshape(1, -1), centroid)[0][0]
        scores.append(score)

    # Sort by score (highest first)
    ranked_indices = np.argsort(scores)[::-1]

    # Pick top sentences, skipping redundant ones
    selected = []
    selected_embeddings = []

    for idx in ranked_indices:
        if len(selected) >= num_sentences:
            break

        # Check if this sentence is too similar to one already selected
        is_redundant = False
        for sel_emb in selected_embeddings:
            sim = cosine_similarity(
                embeddings[idx].reshape(1, -1),
                sel_emb.reshape(1, -1)
            )[0][0]
            if sim > redundancy_threshold:
                is_redundant = True
                break

        if not is_redundant:
            selected.append(idx)
            selected_embeddings.append(embeddings[idx])

    # Return sentences in original order
    selected.sort()
    return [sentences[i] for i in selected]


# Test it
summary = extractive_summary(topic_sentences, num_sentences=2)
print("Summary:")
for s in summary:
    print(f"  {s}")

Summary:
  They consist of layers of interconnected nodes called neurons.
  The key insight is that these networks can learn complex patterns from data.


In [5]:
from nltk.tokenize import sent_tokenize

def generate_topic_title(sentences: list[str], all_topics_text: list[str], max_words: int = 6) -> str:
    """
    Generates a short title for a topic using TF-IDF to find key terms,
    then extracts a phrase from the most representative sentence.
    """
    topic_text = " ".join(sentences)
    
    # TF-IDF across all topics to find what makes this topic unique
    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(all_topics_text)
    feature_names = vectorizer.get_feature_names_out()
    
    # Find this topic's index in all_topics_text
    topic_idx = all_topics_text.index(topic_text)
    scores = tfidf_matrix[topic_idx].toarray()[0]
    
    # Get top keywords for this topic
    top_indices = scores.argsort()[-5:][::-1]
    keywords = set(feature_names[idx] for idx in top_indices)
    
    # Score each sentence by how many keywords it contains
    best_sentence = sentences[0]
    best_count = 0
    for sentence in sentences:
        count = sum(1 for word in sentence.lower().split() if word.strip(".,!?") in keywords)
        if count > best_count:
            best_count = count
            best_sentence = sentence
    
    # Truncate to max_words
    words = best_sentence.split()
    if len(words) > max_words:
        return " ".join(words[:max_words]) + "..."
    return best_sentence


# Test with Steve Jobs topics
topics_sentences = [
    [
        "When I was seventeen, I read a quote that went something like, if you live each day as if it was your last, someday you'll most certainly be right.",
        "It made an impression on me, and since then, for the past thirty-three years, I have looked in the mirror every morning and asked myself, if today were the last day of my life, would I want to do what I am about to do today?",
    ],
    [
        "And whenever the answer has been no for too many days in a row, I know I need to change something.",
        "Remembering that I'll be dead soon is the most important tool I've ever encountered to help me make the big choices in life.",
        "Because almost everything, all external expectations, all pride, all fear of embarrassment or failure, these things just fall away in the face of death, leaving only what is truly important.",
        "Remembering that you are going to die is the best way I know to avoid the trap of thinking you have something to lose.",
    ],
]

# Build the full text for each topic (needed for TF-IDF comparison)
all_topics_text = [" ".join(sents) for sents in topics_sentences]

for i, sentences in enumerate(topics_sentences):
    title = generate_topic_title(sentences, all_topics_text)
    print(f"Topic {i+1}: {title}")

Topic 1: It made an impression on me,...
Topic 2: Remembering that I'll be dead soon...
